In [2]:
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import timedelta

from joblib import dump
from sklearn.preprocessing import StandardScaler

from statsmodels.tsa.stattools import adfuller

In [3]:
# dataset requirements
# timestamp index
# individual label for local series
# variable cols (including 1+ contextual optimizer, e.g. cost)
# target col

dir = './dummy_short_generator/'
fname = 'dummy_short_'
fform = '.csv'
fpath = dir + fname + fform
df0 = (pd.read_csv(fpath)).drop_duplicates()
df0.head()

,timestamp,index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,class,bc_target_sqrt,bc_adj_log_optimizer
0,1997-01-05,15,-2.121371,1.877254,-1.130848,0.185456,5.977733,-1.173733,-0.472260,5.291535,-2.273789,1.419161,-1.318890,5.922892,4.816171,-0.325926,3.119686,0,12671.104404,35.511166
1,1997-01-12,15,-2.121371,1.877254,-1.130848,0.185456,5.977733,-1.173733,-0.472260,5.291535,-2.273789,1.419161,-1.318890,5.922892,4.816171,-0.325926,3.119686,0,22883.898459,57.408747
2,1997-01-19,15,-2.121371,1.877254,-1.130848,0.185456,5.977733,-1.173733,-0.472260,5.291535,-2.273789,1.419161,-1.318890,5.922892,4.816171,-0.325926,3.119686,0,13668.869682,55.202142
3,1997-01-26,11,-0.545852,2.031142,3.868198,-0.592636,-0.171562,0.220793,2.192138,0.764493,1.822855,-1.535099,2.270043,-3.426580,0.441994,-2.463608,-0.854605,10,8315.125011,40.200915
4,1997-01-26,15,-2.121371,1.877254,-1.130848,0.185456,5.977733,-1.173733,-0.472260,5.291535,-2.273789,1.419161,-1.318890,5.922892,4.816171,-0.325926,3.119686,0,29229.704908,32.431929


In [4]:
# flag variables
ts_index = 'timestamp'                    #timestamp index column name
l_label = 'index'                         #local series identifier
variate_cols = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
       '10', '11', '12', '13', '14', 'class',
       'bc_adj_log_optimizer']            #characterizer variables column names
cont_factors = ['bc_adj_log_optimizer']   #contextual factor(s) column names
target_col = 'bc_target_sqrt'             #prediction target column names

#relevant value columns
v_cols = list(set(cont_factors + [target_col]))

In [5]:
# ensure timestamp formating
dtf = '%Y-%m-%d' #string format of datetime index
df0[ts_index] = pd.to_datetime(df0[ts_index], format=dtf)
df0.sort_values(ts_index, ascending=True, inplace=True)

# ensure cont_factors and target are moved last
new_cols = [col for col in df0.columns if col not in v_cols] + v_cols
df0 = df0[new_cols]

In [6]:
# transformation step to accomodate different stationarity states locally

## run augmented dickey-fuller test to control stationarity
def adf_t(col_series, pv=0.05, verbose=False):    
#        col_series = column with series for test
#        pv = p-val threshold
#        autolag = akaike's information criteria
    r = adfuller(col_series, autolag='AIC')
    output = {'stat test':round(r[0], 4), 'pval':round(r[1], 4), 'lags':round(r[2], 4), 'obs':r[3]}
    pv = output['pval']     
    if r[1] <= pv:
        #stationary
        st = 1
    else:
        #non-stationary
        st = 0
    return st

In [7]:
#transform one local series at a time
df_diff = pd.DataFrame()
st_check = 0

#pre-difference base df
df_base = df0.copy()

#max differencing order
md_ord = 6

In [8]:
#iterate over local series
for i in list(set(df_base[l_label])):

    #temporary local df
    df_l = df_base[df_base[l_label]==i].copy()

    df_l_i = df_l[[c for c in df_base.columns if c \
        not in v_cols]].copy()

    df_l_d = df_l[v_cols].copy()

    #local series st check
    st_check = 1
    
    #difference order
    diff_order = 0
    df_l_d['diff_order'] = 0 #placeholder

    while st_check < 2:        
           
        c = 0
        
        #iterate over contextual factors and target col
        while c < len(v_cols):
            
            # column to be checked
            col = v_cols[c]    

            # non stat first
            st_i = 0

            #try adf, assume stationary if failed
            # (will occur for very short, cold-starts, etc.)
            try:
                st_i = adf_t(df_l_d[col])
                if st_i == 1:
                    st_check = 1
                    c = c + 1
            except:
                st_i = 1
                st_check = 1
                c = c + 1
            
            if st_i == 0:
                st_check = 0
                c = 0
                df_l_d = df_l_d.diff().dropna()
                diff_order = diff_order + 1
            
        if st_check == 1 or diff_order > md_ord:

            #if stat after all cols are checked, break cycle
            df_l = pd.concat([df_l_i, df_l_d], axis=1)
            
            #move target to last
            df_l = df_l.reindex(columns = [col for col in df_l.columns \
                                           if col != target_col] + [target_col])
            
            df_l['diff_order'] = diff_order
            
            # continuous append
            df_diff = pd.concat([df_diff, df_l], axis=0, ignore_index=True)  
            st_check = 2
            #break

CPU times: total: 14.1 s
Wall time: 18.8 s


In [9]:
((df_diff.dropna()).drop_duplicates()).to_csv('diff_' + fname + '.csv', index=False)

In [20]:
df_diff.head()

,timestamp,index,0,1,2,3,4,5,6,7,...,9,10,11,12,13,14,class,bc_adj_log_optimizer,diff_order,bc_target_sqrt
0,1997-05-25,0,-2.026,-0.941119,-1.174414,1.565526,1.030992,0.20578,1.913691,-2.059491,...,0.833519,1.637373,1.325719,1.782403,-1.58989,1.928946,9,NaN,28,NaN
1,1997-06-01,0,-2.026,-0.941119,-1.174414,1.565526,1.030992,0.20578,1.913691,-2.059491,...,0.833519,1.637373,1.325719,1.782403,-1.58989,1.928946,9,NaN,28,NaN
2,1997-06-08,0,-2.026,-0.941119,-1.174414,1.565526,1.030992,0.20578,1.913691,-2.059491,...,0.833519,1.637373,1.325719,1.782403,-1.58989,1.928946,9,NaN,28,NaN
3,1997-06-15,0,-2.026,-0.941119,-1.174414,1.565526,1.030992,0.20578,1.913691,-2.059491,...,0.833519,1.637373,1.325719,1.782403,-1.58989,1.928946,9,NaN,28,NaN
4,1997-06-22,0,-2.026,-0.941119,-1.174414,1.565526,1.030992,0.20578,1.913691,-2.059491,...,0.833519,1.637373,1.325719,1.782403,-1.58989,1.928946,9,NaN,28,NaN


In [ ]:
# data scaling
scaler = StandardScaler()
scaler.fit(((df_diff[[c for c in \
                df_diff.columns if c != l_label]]).set_index(ts_index)).to_numpy())

dump(scaler, 'scaler_' + fname + '.joblib')

np.save('scaler_stats_' + fname + '.npy', np.array([scaler.mean_, scaler.scale_]))

['scaler_dummy_short_.joblib']